In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import KNNImputer

In [30]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_test['Transported'] = False
df = pd.concat([df_train, df_test], sort = False)
df.drop(['Name', 'PassengerId'], axis = 1, inplace = True)
df.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [31]:
df.shape[0] == df_train.shape[0] + df_test.shape[0]

True

In [32]:
df.isnull().sum()

HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
dtype: int64

In [33]:
# - Data Cleaning

In [34]:
df[['Deck', 'Number', 'Side' ]] = df['Cabin'].str.split('/', expand = True)
df = df.drop(columns = ['Cabin'])
df['Deck'] = df['Deck'].fillna('U')
df['Number'] = df['Number'].fillna('-1')
df['Side'] = df['Side'].fillna('U')

In [35]:
# after spliting Cabin

df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Number,Side
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,B,0,P
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,F,0,S
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,A,0,S
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,A,0,S
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,F,1,S


In [36]:
df['Side'].value_counts()

Side
S    6381
P    6290
U     299
Name: count, dtype: int64

In [37]:
df['Deck'] = df['Deck'].map({'F' : 0, 'G' : 1, 'E' : 2, 'B' : 3, 'C' : 4, 'D' : 5, 'A' : 6, 'U' : 7, 'T' : 8})
df['Side'] = df['Side'].map({'U' : -1, 'P' : 1, 'S' : 2})




In [38]:
impute_list = ['Age', 'VIP', 'Number', 'CryoSleep', 'Side', 'Deck', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
rest = list(set(df.columns) - set(impute_list))

# Pass the variable 'rest' directly without quotes
df_rest = df[rest]


knn = KNNImputer(n_neighbors=5)

# Convert the NumPy array output back into a Pandas DataFrame
df_imputed = pd.DataFrame(knn.fit_transform(df[impute_list]), columns=impute_list)

# Concatenate the two DataFrames
df = pd.concat([df_rest.reset_index(drop=True), df_imputed], axis=1)

In [39]:
df['Destination'] = df['Destination'].fillna('Un')
df['HomePlanet'] = df['HomePlanet'].fillna('U')

# One-hot encoding - for categorical columns HomePlanet and destination
catg_cols = ['HomePlanet', 'Destination']

for cols in catg_cols:
    df = pd.concat([df, pd.get_dummies(df[cols], prefix = cols)], axis =1 )

df= df.drop(columns = catg_cols)

In [40]:
df.isnull().sum()


Transported                  0
Age                          0
VIP                          0
Number                       0
CryoSleep                    0
Side                         0
Deck                         0
RoomService                  0
FoodCourt                    0
ShoppingMall                 0
Spa                          0
VRDeck                       0
HomePlanet_Earth             0
HomePlanet_Europa            0
HomePlanet_Mars              0
HomePlanet_U                 0
Destination_55 Cancri e      0
Destination_PSO J318.5-22    0
Destination_TRAPPIST-1e      0
Destination_Un               0
dtype: int64

In [41]:
df.head()

,Transported,Age,VIP,Number,CryoSleep,Side,Deck,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,HomePlanet_U,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_Un
0,False,39.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,True,False
1,True,24.0,0.0,0.0,0.0,2.0,0.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,True,False
2,False,58.0,1.0,0.0,0.0,2.0,6.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,True,False
3,False,33.0,0.0,0.0,0.0,2.0,6.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,True,False
4,True,16.0,0.0,1.0,0.0,2.0,0.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,True,False


In [86]:
bill_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df['amt_spend'] = df[bill_cols].sum(axis = 1)
df['std_amt_spend'] = df[bill_cols].std(axis = 1)
df['mean_amt_spend'] = df[bill_cols].mean(axis = 1)

df['5_high_cols'] = df['CryoSleep']+df['HomePlanet_Europa']+ df['Destination_55 Cancri e'] + df['Deck'] + df['Side']
df['5_low_cols'] = df['mean_amt_spend'] + df['amt_spend'] + df['HomePlanet_Earth'] + df['std_amt_spend'] + df['VRDeck']

In [87]:
df.corr()['Transported'].sort_values(ascending = False)

Transported                  1.000000
CryoSleep                    0.324480
5_high_cols                  0.169998
4_high_cols                  0.151830
HomePlanet_Europa            0.131977
Destination_55 Cancri e      0.083625
Deck                         0.062790
Side                         0.059872
6_high_cols                  0.035013
FoodCourt                    0.034771
HomePlanet_U                 0.006403
HomePlanet_Mars              0.005643
ShoppingMall                 0.004163
Destination_PSO J318.5-22    0.000760
Destination_Un              -0.000554
VIP                         -0.018644
Number                      -0.035240
Age                         -0.050604
Destination_TRAPPIST-1e     -0.072731
HomePlanet_Earth            -0.119644
std_amt_spend               -0.121149
amt_spend                   -0.140423
mean_amt_spend              -0.140423
VRDeck                      -0.142783
4_low_cols                  -0.148782
5_low_cols                  -0.148782
Spa         

In [88]:
!pip install lightgbm

In [89]:
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [90]:
df_train, df_test = df.iloc[:df_train.shape[0]], df.iloc[df_train.shape[0]:]
df_test = df_test.drop(columns='Transported')
df_train.shape, df_test.shape

((8693, 29), (4277, 28))

In [91]:
x = df_train.drop(columns = 'Transported')
y = df_train['Transported']

In [92]:
x_train, x_test, y_train,  y_test = train_test_split(x, y, random_state = 42, test_size = 0.2)

In [93]:
model_1 = LogisticRegression()
model_2 = DecisionTreeClassifier()
model_3 = RandomForestClassifier()
model_4 = XGBClassifier()
model_5 = LGBMClassifier()


In [94]:
# Model_1 test

model_1.fit(x_train, y_train)
y_pred = model_1.predict(x_test)
accuracy_score(y_test, y_pred)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7705577918343876

In [95]:
# Model_2 test

model_2.fit(x_train, y_train)
y_pred = model_2.predict(x_test)
accuracy_score(y_test, y_pred)

0.7406555491661875

In [96]:
# Model31 test

model_3.fit(x_train, y_train)
y_pred = model_3.predict(x_test)
accuracy_score(y_test, y_pred)

0.7878090856814262

In [97]:
# Model_4 test

model_4.fit(x_train, y_train)
y_pred = model_4.predict(x_test)
accuracy_score(y_test, y_pred)

0.7889591719378953

In [98]:
# Model_5 test

model_5.fit(x_train, y_train)
y_pred = model_5.predict(x_test)
accuracy_score(y_test, y_pred)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3500, number of negative: 3454
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001104 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3508
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503307 -> initscore=0.013230
[LightGBM] [Info] Start training from score 0.013230


0.8096607245543416

In [99]:
# Final Output: Prepare

df_dummy = pd.read_csv('test.csv')
pred = model_5.predict(df_test)

final = pd.DataFrame()
final['PassengerId'] = df_dummy['PassengerId']
final['Transported'] = pred
final.to_csv('submission.csv', index = False)